# Strands Decider Example

This example uses `DeciderBackend` with [Strands Decider](https://github.com/strands-labs/strands-decider), a 2B open-weights decision model. Decision models do not generate text: each eval mode is answered with a probability read off the model, so `score` and `confidence` are measured, not self-reported, and `reasoning` is always `None`.

Start a local server first (CUDA, Apple Silicon or CPU):

```bash
pip install grounded-ai[decider]
strands-decider serve StrandsAgents/strands-decider-2B-hobson-v19 --port 8000
```

In [1]:
import json
import os
from pprint import pprint

from grounded_ai import Evaluator

BASE_URL = os.getenv("DECIDER_BASE_URL", "http://127.0.0.1:8000")

In [2]:
# Uses the 'decider/' prefix to trigger auto-detection
evaluator = Evaluator("decider/strands-decider-2B-hobson-v19", base_url=BASE_URL)

context = """
The Apollo 11 mission landed the first humans on the Moon.
Neil Armstrong and Buzz Aldrin walked on the lunar surface.
Michael Collins remained in orbit in the Command Module.
"""

# Test Case 1: Accurate
query = "Who stayed in orbit?"
response_accurate = "Michael Collins remained in orbit."

In [3]:
result = evaluator.evaluate(query=query, context=context, response=response_accurate)

pprint(json.loads(result.model_dump_json()))

{'confidence': 0.8538, 'label': 'faithful', 'reasoning': None, 'score': 0.0731}


In [4]:
# Test Case 2: Inaccurate
response_inaccurate = "Buzz Aldrin stayed in the orbiter while Neil went down alone."

result = evaluator.evaluate(query=query, context=context, response=response_inaccurate)

pprint(json.loads(result.model_dump_json()))

{'confidence': 0.3193999999999999,
 'label': 'hallucination',
 'reasoning': None,
 'score': 0.6597}


In [5]:
# Other eval modes: TOXICITY and RAG_RELEVANCE
toxicity = Evaluator("decider/strands-decider-2B-hobson-v19", base_url=BASE_URL, eval_mode="TOXICITY")

for text in ["You are a worthless idiot and everyone hates you.", "Thanks for the quick reply, that fixed it."]:
    result = toxicity.evaluate(response=text)
    print(f"{result.label:10s} score={result.score:.3f} confidence={result.confidence:.3f}  {text}")

toxic      score=0.983 confidence=0.967  You are a worthless idiot and everyone hates you.
non-toxic  score=0.041 confidence=0.917  Thanks for the quick reply, that fixed it.


## Custom Schemas

An output schema maps one-to-one onto what the model returns. Every field becomes one question, all answered in a single request, and each field is exactly one of the three question types:

- `noul`: `bool`, or a `float` bounded to `[0, 1]`
- `choice`: `Literal` / `Enum`
- `score`: an `int` or `float` with `ge`/`le` bounds (2 to 10 values)

The field's `description` is the question (the field name is used when there is none). Any other required field type, such as `str`, is refused. There is no system message: `system_prompt` is not sent.

In [6]:
from typing import Literal

from pydantic import BaseModel, Field


class Ticket(BaseModel):
    urgent: bool = Field(description="Does this need a reply within the hour?")
    area: Literal["billing", "bug", "account", "other"] = Field(description="Which team owns it?")
    p_harmful: float = Field(ge=0, le=1, description="Is this request harmful?")


result = evaluator.evaluate(
    response="You have charged me twice and my account is now overdrawn. I need this reversed today.",
    output_schema=Ticket,
)
print(result)

urgent=True area='billing' p_harmful=0.3856


### Ratings (`score`)

An `int` or `float` with `ge`/`le` bounds is asked as a rating over those levels. An `int` gets the most probable level, a `float` the expected value.

In [ ]:
class Quality(BaseModel):
    clarity: int = Field(ge=1, le=5, description="Rate how clearly the problem is described.")
    politeness: float = Field(ge=1, le=5, description="Rate how polite the message is.")
    tone: Literal["calm", "frustrated", "angry"]  # no description: the field name is the question


result = evaluator.evaluate(
    response="You have charged me twice and my account is now overdrawn. I need this reversed today.",
    output_schema=Quality,
)
print(result)

## Custom Evaluations

An evaluation of your own is an output schema, the same way as on the OpenAI and Anthropic backends: subclass `EvaluationOutput` and make `label` a `Literal` of your options. The field's `description` is the question, and `score` is the probability of the first option.

The built-in modes (`HALLUCINATION`, `TOXICITY`, `RAG_RELEVANCE`) can also be picked for a single call.

In [ ]:
from grounded_ai import EvaluationOutput


class BrandVoice(EvaluationOutput):
    label: Literal["on-brand", "off-brand"] = Field(
        description="Is the response written in our brand voice? "
        "on-brand: friendly, plain words, no jargon. off-brand: formal, salesy or full of jargon."
    )


for text in [
    "No worries, we've refunded you. It should show up in a few days.",
    "Pursuant to our policy, remuneration will be processed in due course.",
]:
    result = evaluator.evaluate(response=text, output_schema=BrandVoice)
    print(f"{result.label:10s} score={result.score:.3f}  {text}")

# A built-in mode for one call, without changing the evaluator
print(evaluator.evaluate(response="Thanks, that fixed it.", eval_mode="toxicity").label)

## Custom Inputs

The model reads a `state`, which is either text or a JSON object. Any Pydantic input model works: its fields are sent as an object, or its own `formatted_prompt` / a custom `base_template` is rendered and sent as text.

In [ ]:
from grounded_ai import EvaluationInput

class RuleCheck(EvaluationOutput):
    label: Literal["violates", "follows"] = Field(description="Does the text follow the rule?")


# 1. A template for one call
print(evaluator.evaluate(
    response="The API endpoint defaults to port 8080.",
    base_template="Rule: services must only listen on port 443.\nText: {{ response }}",
    output_schema=RuleCheck,
).label)


# 2. A template as a reusable input class
class PortPolicy(EvaluationInput):
    base_template: str = "Rule: services must only listen on port 443.\nText: {{ response }}"


print(evaluator.evaluate(PortPolicy(response="The service listens on 443."), output_schema=RuleCheck).label)


# 3. Your own input model: fields are sent as a JSON object
class SupportTurn(BaseModel):
    customer_message: str
    agent_reply: str
    account_tier: Literal["free", "pro"]


class ReplyCheck(BaseModel):
    answers_the_question: bool
    apologizes: bool


print(evaluator.evaluate(
    SupportTurn(
        customer_message="Why was I charged twice?",
        agent_reply="Sorry about that! The second charge was a duplicate and is refunded.",
        account_tier="pro",
    ),
    output_schema=ReplyCheck,
))

## Code Review: the same example as the Anthropic and OpenAI notebooks

Same input schema, same call, no mode. The **output schema is different**, because a decision model picks answers and does not write text:

| Anthropic / OpenAI notebook | Here | Why |
| :--- | :--- | :--- |
| `is_bug_free: bool` | `is_bug_free: bool` | yes/no (`noul`) |
| `security_risk: bool` | `p_security_risk: float` in `[0, 1]` | the probability itself instead of a yes/no (`noul`) |
| `complexity_score: int`, range only in the description | `complexity_score: int` with `ge=1, le=10` | a rating needs real bounds (`score`) |
| `suggestions: List[str]` | removed | free text cannot be generated |
| (none) | `severity: Literal["low", "medium", "high"]` | pick one option (`choice`) |

In [ ]:
# 1. Input schema: identical to the Anthropic/OpenAI notebooks
class CodeReviewInput(BaseModel):
    code: str
    language: str


# 2. Output schema: every field is one decider answer type (noul, choice or score)
class CodeReviewDecision(BaseModel):
    is_bug_free: bool                                                # noul   -> True when p >= threshold
    p_security_risk: float = Field(ge=0, le=1)                       # noul   -> the probability itself
    severity: Literal["low", "medium", "high"]                       # choice -> the option picked
    complexity_score: int = Field(ge=1, le=10, description="Rate the complexity of the code.")  # score -> 1-10
    # suggestions: List[str]  <- not possible: a decision model does not write text


# DANGEROUS: SQL Injection vulnerability
code_sample = """
def execute_query(user_input):
    query = f"SELECT * FROM users WHERE name = '{user_input}'"
    db.execute(query)
"""
review_input = CodeReviewInput(code=code_sample, language="python")

# 3. Same call as on the other backends
review_result = evaluator.evaluate(input_data=review_input, output_schema=CodeReviewDecision)

pprint(json.loads(review_result.model_dump_json()))